In [5]:
import pandas as pd
import numpy as np
from scipy import stats
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import pyterrier as pt
if not pt.started():
    pt.init()

# 设置pandas显示选项
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)
pd.set_option('display.precision', 4)

print("=" * 80)
print("完整QPP指标计算与PRF预测分析")
print("=" * 80)

# ========== 可调参数 ==========
# 调整这些阈值来控制样本量和标签质量
BENEFIT_THRESHOLD = 0.52  # PRF-Benefit 的阈值（建议范围: 0.52-0.60）
HURT_THRESHOLD = 0.48     # PRF-Hurt 的阈值（建议范围: 0.40-0.48）
# 更严格的阈值 (如 0.60/0.40): 样本少但标签质量高
# 更宽松的阈值 (如 0.52/0.48): 样本多但可能包含模糊案例

print(f"\n📊 当前阈值设置:")
print(f"   PRF-Benefit: b_preference_ratio > {BENEFIT_THRESHOLD}")
print(f"   PRF-Hurt:    b_preference_ratio < {HURT_THRESHOLD}")
print(f"   提示: 调整阈值来平衡样本量和标签质量")
print("=" * 80)

# ========== 1. 加载检索结果 ==========
print("\n加载检索结果...")

df_colbert = pt.io.read_results("../../ColBERT-PRF-VirtualAppendix/ColBERT E2E/E2E.2019.res.gz")
df_colbert_prf = pt.io.read_results("../../ColBERT-PRF-VirtualAppendix/ColBERT-PRF/MSMARCO Passage Res/ColBERT PRF Ranker_beta=1/prf_rank_beta1.2019.res.gz")

# 确保qid是字符串类型（与其他数据保持一致）
df_colbert['qid'] = df_colbert['qid'].astype(str)
df_colbert_prf['qid'] = df_colbert_prf['qid'].astype(str)

print(f"✓ ColBERT E2E: {len(df_colbert)} 条结果, {df_colbert['qid'].nunique()} 个查询")
print(f"✓ ColBERT PRF: {len(df_colbert_prf)} 条结果, {df_colbert_prf['qid'].nunique()} 个查询")

# 加载preference数据
preference_paths = ['preference.csv']
preference_df = None
for path in preference_paths:
    try:
        preference_df = pd.read_csv(path)
        # 确保qid是字符串类型
        preference_df['qid'] = preference_df['qid'].astype(str)
        print(f"✓ 加载 preference 数据: {path} ({len(preference_df)} 条)")
        break
    except FileNotFoundError:
        continue

if preference_df is None:
    print("错误: 找不到 preference.csv 文件")
    exit(1)

# 加载Dense QPP（如果有）
dense_qpp_df = None
dense_paths = ['dense_qpp_scores.csv', 'rbo_Noisyq_0.01.dl2019_queries.tsv.tsv_500']
for path in dense_paths:
    try:
        # 尝试不同的分隔符
        try:
            dense_qpp_df = pd.read_csv(path, sep='\t')
        except:
            dense_qpp_df = pd.read_csv(path)
        
        # 检查并调整列名
        print(f"  原始列名: {dense_qpp_df.columns.tolist()}")
        
        if len(dense_qpp_df.columns) >= 2:
            # 假设第一列是qid，第二列是score
            dense_qpp_df = dense_qpp_df.iloc[:, :2]
            dense_qpp_df.columns = ['qid', 'dense_qpp']
            # 确保qid是字符串类型
            dense_qpp_df['qid'] = dense_qpp_df['qid'].astype(str)
        
        print(f"✓ 加载 Dense QPP: {path} ({len(dense_qpp_df)} 条)")
        break
    except FileNotFoundError:
        continue
    except Exception as e:
        print(f"  警告: 加载 {path} 失败: {e}")
        continue

# ========== 2. 计算各种QPP指标 ==========
print("\n" + "=" * 80)
print("计算QPP指标...")
print("=" * 80)

def calculate_nqc(df, group_by='qid', score_col='score'):
    """
    NQC (Normalized Query Commitment) = std(scores) / mean(scores)
    衡量检索分数的离散程度
    """
    nqc_results = []
    for qid, group in df.groupby(group_by):
        scores = group[score_col].values
        mean_score = np.mean(scores)
        std_score = np.std(scores)
        nqc = std_score / mean_score if mean_score != 0 else 0
        nqc_results.append({'qid': qid, 'nqc': nqc})
    return pd.DataFrame(nqc_results)

def calculate_wig(df_baseline, df_prf, group_by='qid', score_col='score', top_k=10):
    """
    WIG (Weighted Information Gain)
    衡量PRF后top-k文档分数的提升
    WIG = (1/k) * Σ(score_prf - score_baseline)
    """
    wig_results = []
    for qid in df_baseline[group_by].unique():
        baseline_group = df_baseline[df_baseline[group_by] == qid].head(top_k)
        prf_group = df_prf[df_prf[group_by] == qid].head(top_k)
        
        if len(baseline_group) == 0 or len(prf_group) == 0:
            continue
        
        # 计算平均分数差
        baseline_mean = baseline_group[score_col].mean()
        prf_mean = prf_group[score_col].mean()
        wig = prf_mean - baseline_mean
        
        wig_results.append({'qid': qid, 'wig': wig})
    
    return pd.DataFrame(wig_results)

def calculate_uef(df, group_by='qid', score_col='score', top_k=10):
    """
    UEF (Utility Estimation Framework)
    基于top-k文档分数的方差和均值
    UEF = mean(top_k_scores) * (1 - std(top_k_scores))
    """
    uef_results = []
    for qid, group in df.groupby(group_by):
        top_scores = group.head(top_k)[score_col].values
        if len(top_scores) == 0:
            continue
        
        mean_score = np.mean(top_scores)
        std_score = np.std(top_scores)
        uef = mean_score * (1 - std_score) if std_score < 1 else 0
        
        uef_results.append({'qid': qid, 'uef': uef})
    
    return pd.DataFrame(uef_results)

def calculate_smv(df, group_by='qid', score_col='score', top_k=10):
    """
    SMV (Standard deviation and Mean Variance)
    衡量top-k文档分数的统计特性
    SMV = std(top_k_scores) / sqrt(k)
    """
    smv_results = []
    for qid, group in df.groupby(group_by):
        top_scores = group.head(top_k)[score_col].values
        if len(top_scores) == 0:
            continue
        
        std_score = np.std(top_scores)
        smv = std_score / np.sqrt(len(top_scores))
        
        smv_results.append({'qid': qid, 'smv': smv})
    
    return pd.DataFrame(smv_results)

# 计算各个指标
print("\n1. 计算 NQC (基于E2E结果)...")
nqc_df = calculate_nqc(df_colbert)
print(f"   ✓ 计算完成: {len(nqc_df)} 个查询")

print("\n2. 计算 WIG (E2E vs PRF差异)...")
wig_df = calculate_wig(df_colbert, df_colbert_prf)
print(f"   ✓ 计算完成: {len(wig_df)} 个查询")

print("\n3. 计算 UEF (基于E2E结果)...")
uef_df = calculate_uef(df_colbert)
print(f"   ✓ 计算完成: {len(uef_df)} 个查询")

print("\n4. 计算 SMV (基于E2E结果)...")
smv_df = calculate_smv(df_colbert)
print(f"   ✓ 计算完成: {len(smv_df)} 个查询")

# 保存中间结果
nqc_df.to_csv('nqc_scores.csv', index=False)
wig_df.to_csv('wig_scores.csv', index=False)
uef_df.to_csv('uef_scores.csv', index=False)
smv_df.to_csv('smv_scores.csv', index=False)
print("\n✓ QPP分数已保存")

# ========== 3. 合并所有数据 ==========
print("\n" + "=" * 80)
print("合并数据...")
print("=" * 80)

merged_df = preference_df[['qid', 'preference', 'b_preference_ratio']].copy()
print(f"初始 preference 数据: {len(merged_df)} 条查询")
print(f"  - qid 类型: {merged_df['qid'].dtype}")
print(f"  - qid 样例: {merged_df['qid'].head(3).tolist()}")

# 合并各QPP指标
print(f"\n合并 NQC...")
print(f"  - NQC 数据: {len(nqc_df)} 条, qid类型: {nqc_df['qid'].dtype}")
print(f"  - NQC qid 样例: {nqc_df['qid'].head(3).tolist()}")
merged_df = pd.merge(merged_df, nqc_df, on='qid', how='left')
print(f"  - 合并后: {len(merged_df)} 条, 有效 NQC: {merged_df['nqc'].notna().sum()}")

print(f"\n合并 WIG...")
print(f"  - WIG 数据: {len(wig_df)} 条, qid类型: {wig_df['qid'].dtype}")
merged_df = pd.merge(merged_df, wig_df, on='qid', how='left')
print(f"  - 合并后: {len(merged_df)} 条, 有效 WIG: {merged_df['wig'].notna().sum()}")

print(f"\n合并 UEF...")
print(f"  - UEF 数据: {len(uef_df)} 条, qid类型: {uef_df['qid'].dtype}")
merged_df = pd.merge(merged_df, uef_df, on='qid', how='left')
print(f"  - 合并后: {len(merged_df)} 条, 有效 UEF: {merged_df['uef'].notna().sum()}")

print(f"\n合并 SMV...")
print(f"  - SMV 数据: {len(smv_df)} 条, qid类型: {smv_df['qid'].dtype}")
merged_df = pd.merge(merged_df, smv_df, on='qid', how='left')
print(f"  - 合并后: {len(merged_df)} 条, 有效 SMV: {merged_df['smv'].notna().sum()}")

if dense_qpp_df is not None:
    print(f"\n合并 Dense QPP...")
    print(f"  - Dense QPP 数据: {len(dense_qpp_df)} 条, qid类型: {dense_qpp_df['qid'].dtype}")
    print(f"  - Dense QPP qid 样例: {dense_qpp_df['qid'].head(3).tolist()}")
    merged_df = pd.merge(merged_df, dense_qpp_df, on='qid', how='left')
    print(f"  - 合并后: {len(merged_df)} 条, 有效 Dense QPP: {merged_df['dense_qpp'].notna().sum()}")

print(f"\n合并后总数: {len(merged_df)} 条查询")

# 过滤：使用 b_preference_ratio 定义 PRF 效果
print(f"\n定义 PRF 效果的阈值:")
print(f"  - PRF-Benefit: b_preference_ratio > {BENEFIT_THRESHOLD}")
print(f"  - PRF-Hurt: b_preference_ratio < {HURT_THRESHOLD}")
print(f"  - Neutral: {HURT_THRESHOLD} <= b_preference_ratio <= {BENEFIT_THRESHOLD}")

# 创建新的 preference 标签
def classify_prf_effect(ratio):
    if pd.isna(ratio) or ratio == 0:
        return 'Insufficient-Data'
    elif ratio > BENEFIT_THRESHOLD:
        return 'PRF-Benefit'
    elif ratio < HURT_THRESHOLD:
        return 'PRF-Hurt'
    else:
        return 'Neutral'

merged_df['preference_new'] = merged_df['b_preference_ratio'].apply(classify_prf_effect)

print(f"\n重新分类后的分布:")
print(merged_df['preference_new'].value_counts())

# 只保留有明确倾向的查询
merged_df = merged_df[merged_df['preference_new'].isin(['PRF-Benefit', 'PRF-Hurt'])].copy()
print(f"\n过滤后: {len(merged_df)} 条查询 (Benefit: {(merged_df['preference_new']=='PRF-Benefit').sum()}, "
      f"Hurt: {(merged_df['preference_new']=='PRF-Hurt').sum()})")

# 创建二分类标签
merged_df['prf_benefit'] = (merged_df['preference_new'] == 'PRF-Benefit').astype(int)

# 统计可用性
print("\nQPP指标可用性:")
qpp_columns = ['nqc', 'wig', 'uef', 'smv']
if dense_qpp_df is not None:
    qpp_columns.append('dense_qpp')

available_qpps = []
for col in qpp_columns:
    if col in merged_df.columns:
        valid_count = merged_df[col].notna().sum()
        coverage = valid_count / len(merged_df) * 100
        print(f"  {col.upper()}: {valid_count}/{len(merged_df)} ({coverage:.1f}%)")
        if coverage >= 50:
            available_qpps.append(col)

print(f"\n可用于分析的QPP指标: {', '.join([q.upper() for q in available_qpps])}")

# ========== 4. 为每个QPP找最佳阈值 ==========
print("\n" + "=" * 80)
print("寻找最佳预测阈值 (基于F1-score)")
print("=" * 80)

results = []

for qpp_col in available_qpps:
    # 获取有效数据
    valid_data = merged_df[[qpp_col, 'prf_benefit']].dropna()
    if len(valid_data) < 10:
        print(f"\n{qpp_col.upper()}: 数据不足，跳过")
        continue
    
    scores = valid_data[qpp_col].values
    labels = valid_data['prf_benefit'].values
    
    # 尝试不同阈值
    percentiles = np.linspace(10, 90, 17)
    thresholds = [np.percentile(scores, p) for p in percentiles]
    
    best_f1 = 0
    best_threshold = None
    best_metrics = None
    best_direction = None
    
    # 尝试两个方向的预测
    for direction in ['higher_better', 'lower_better']:
        for threshold in thresholds:
            if direction == 'higher_better':
                # 假设：高QPP分数 = 应该使用PRF
                predictions = (scores >= threshold).astype(int)
            else:
                # 假设：低QPP分数 = 应该使用PRF
                predictions = (scores <= threshold).astype(int)
            
            # 计算指标
            acc = accuracy_score(labels, predictions)
            prec = precision_score(labels, predictions, zero_division=0)
            rec = recall_score(labels, predictions, zero_division=0)
            f1 = f1_score(labels, predictions, zero_division=0)
            
            if f1 > best_f1:
                best_f1 = f1
                best_threshold = threshold
                best_direction = direction
                best_metrics = {
                    'accuracy': acc,
                    'precision': prec,
                    'recall': rec,
                    'f1': f1
                }
    
    # 计算AUC-ROC
    try:
        # 对于lower_better的情况，需要反转分数
        if best_direction == 'lower_better':
            auc = roc_auc_score(labels, -scores)
        else:
            auc = roc_auc_score(labels, scores)
    except:
        auc = 0.5
    
    results.append({
        'QPP': qpp_col.upper(),
        'direction': '越高越好' if best_direction == 'higher_better' else '越低越好',
        'threshold': best_threshold,
        'accuracy': best_metrics['accuracy'],
        'precision': best_metrics['precision'],
        'recall': best_metrics['recall'],
        'f1': best_metrics['f1'],
        'auc_roc': auc,
        'n_samples': len(valid_data)
    })
    
    print(f"\n{qpp_col.upper()}:")
    print(f"  预测策略: {best_direction}")
    print(f"  最佳阈值: {best_threshold:.4f}")
    print(f"  Accuracy:  {best_metrics['accuracy']:.4f}")
    print(f"  Precision: {best_metrics['precision']:.4f}")
    print(f"  Recall:    {best_metrics['recall']:.4f}")
    print(f"  F1-score:  {best_metrics['f1']:.4f}")
    print(f"  AUC-ROC:   {auc:.4f}")

# ========== 5. 结果对比 ==========
results_df = pd.DataFrame(results)
results_df = results_df.sort_values('f1', ascending=False)

print("\n" + "=" * 80)
print("QPP METHOD RANKING (Sorted by F1-Score)")
print("=" * 80)
print(results_df.to_string(index=False))

# 保存结果
results_df.to_csv('qpp_prediction_results.csv', index=False)
merged_df.to_csv('qpp_merged_data.csv', index=False)
print("\n✓ Results saved: qpp_prediction_results.csv")
print("✓ Full data saved: qpp_merged_data.csv")

# ========== 6. 详细结果表格与指标说明 ==========
print("\n" + "=" * 80)
print("DETAILED PERFORMANCE METRICS")
print("=" * 80)

# 创建更详细的表格
detailed_results = []
for qpp_col in available_qpps:
    valid_data = merged_df[[qpp_col, 'prf_benefit']].dropna()
    if len(valid_data) < 10:
        continue
    
    scores = valid_data[qpp_col].values
    labels = valid_data['prf_benefit'].values
    
    # 获取已经计算的最佳结果
    qpp_result = results_df[results_df['QPP'] == qpp_col.upper()]
    if len(qpp_result) == 0:
        continue
    
    # 计算额外的统计信息
    benefit_scores = valid_data[valid_data['prf_benefit'] == 1][qpp_col]
    hurt_scores = valid_data[valid_data['prf_benefit'] == 0][qpp_col]
    
    detailed_results.append({
        'QPP_Method': qpp_col.upper(),
        'Strategy': qpp_result['direction'].values[0],
        'Threshold': qpp_result['threshold'].values[0],
        'F1_Score': qpp_result['f1'].values[0],
        'Accuracy': qpp_result['accuracy'].values[0],
        'Precision': qpp_result['precision'].values[0],
        'Recall': qpp_result['recall'].values[0],
        'AUC_ROC': qpp_result['auc_roc'].values[0],
        'N_Samples': len(valid_data),
        'Benefit_Mean': benefit_scores.mean(),
        'Benefit_Std': benefit_scores.std(),
        'Hurt_Mean': hurt_scores.mean(),
        'Hurt_Std': hurt_scores.std(),
        'Mean_Diff': benefit_scores.mean() - hurt_scores.mean()
    })

detailed_df = pd.DataFrame(detailed_results)
detailed_df = detailed_df.sort_values('F1_Score', ascending=False)

print("\nTable 1: Performance Comparison of QPP Methods")
print("-" * 120)
performance_table = detailed_df[['QPP_Method', 'Strategy', 'F1_Score', 'Accuracy', 
                                  'Precision', 'Recall', 'AUC_ROC', 'N_Samples']]
print(performance_table.to_string(index=False))

print("\n\nTable 2: Score Statistics by PRF Effect")
print("-" * 120)
stats_table = detailed_df[['QPP_Method', 'Benefit_Mean', 'Benefit_Std', 
                           'Hurt_Mean', 'Hurt_Std', 'Mean_Diff', 'Threshold']]
print(stats_table.to_string(index=False))

# 保存详细表格
detailed_df.to_csv('qpp_detailed_results.csv', index=False)
print("\n✓ Detailed results saved: qpp_detailed_results.csv")

# ========== 6.5 指标说明 ==========
print("\n" + "=" * 80)
print("METRIC DEFINITIONS & INTERPRETATIONS")
print("=" * 80)

print("""
📊 QPP METHODS (Query Performance Predictors):
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
1. NQC (Normalized Query Commitment)
   - Formula: std(retrieval_scores) / mean(retrieval_scores)
   - Meaning: Measures the divergence of retrieval scores
   - High NQC → More discriminative retrieval, scores vary widely
   - Low NQC → Less discriminative, similar scores across documents

2. WIG (Weighted Information Gain)
   - Formula: mean(PRF_scores) - mean(Baseline_scores)
   - Meaning: Score improvement after PRF on top-k documents
   - Positive WIG → PRF increases scores (potentially beneficial)
   - Negative WIG → PRF decreases scores (potentially harmful)

3. UEF (Utility Estimation Framework)
   - Formula: mean(top_k_scores) × (1 - std(top_k_scores))
   - Meaning: Combines mean score and score consistency
   - High UEF → High scores with low variance (confident retrieval)
   - Low UEF → Either low scores or high variance (uncertain)

4. SMV (Standard deviation and Mean Variance)
   - Formula: std(top_k_scores) / sqrt(k)
   - Meaning: Normalized score variance among top results
   - High SMV → High uncertainty in top results
   - Low SMV → Consistent scores in top results

5. DENSE_QPP (Dense Retrieval Quality Predictor)
   - Meaning: Neural-based query difficulty predictor
   - High score → Query is predicted to be easier/more reliable
   - Low score → Query is predicted to be harder/less reliable

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

📈 EVALUATION METRICS:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
• F1-Score (Primary Metric)
  Range: [0, 1], Higher is better
  Harmonic mean of Precision and Recall
  - > 0.7: Excellent prediction
  - 0.6-0.7: Good prediction
  - 0.5-0.6: Moderate prediction
  - < 0.5: Poor prediction (worse than random with optimal threshold)

• Accuracy
  Range: [0, 1], Higher is better
  Percentage of correct predictions
  Note: Can be misleading with imbalanced classes

• Precision
  Range: [0, 1], Higher is better
  Of queries predicted as "PRF-Benefit", how many truly benefit?
  High precision → Few false positives (won't wrongly use PRF)

• Recall
  Range: [0, 1], Higher is better
  Of queries that truly benefit from PRF, how many did we identify?
  High recall → Few false negatives (won't miss PRF opportunities)

• AUC-ROC (Area Under ROC Curve)
  Range: [0, 1], Higher is better
  Discrimination ability regardless of threshold
  - 1.0: Perfect discrimination
  - 0.5: Random guess (baseline)
  - < 0.5: Worse than random (predictions inverted)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

🎯 PREDICTION STRATEGY:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
• "Higher is Better" (越高越好)
  If QPP score > threshold → Predict PRF-Benefit (use PRF)
  If QPP score ≤ threshold → Predict PRF-Hurt (don't use PRF)
  Example: High WIG means PRF improved scores, so use PRF

• "Lower is Better" (越低越好)
  If QPP score < threshold → Predict PRF-Benefit (use PRF)
  If QPP score ≥ threshold → Predict PRF-Hurt (don't use PRF)
  Example: Low NQC means unclear retrieval, PRF might help

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
""")

# ========== 7. 统计检验 ==========
print("\n" + "=" * 80)
print("STATISTICAL SIGNIFICANCE TESTS (PRF-Benefit vs PRF-Hurt)")
print("=" * 80)

print("\nTable 3: T-test Results for Score Differences")
print("-" * 120)

significance_results = []
for qpp_col in available_qpps:
    benefit_data = merged_df[merged_df['preference_new'] == 'PRF-Benefit'][qpp_col].dropna()
    hurt_data = merged_df[merged_df['preference_new'] == 'PRF-Hurt'][qpp_col].dropna()
    
    if len(benefit_data) > 0 and len(hurt_data) > 0:
        t_stat, p_value = stats.ttest_ind(benefit_data, hurt_data)
        mean_diff = benefit_data.mean() - hurt_data.mean()
        # Cohen's d effect size
        pooled_std = np.sqrt((benefit_data.std()**2 + hurt_data.std()**2) / 2)
        effect_size = mean_diff / pooled_std if pooled_std > 0 else 0
        
        significance_results.append({
            'QPP_Method': qpp_col.upper(),
            'Benefit_Mean': benefit_data.mean(),
            'Hurt_Mean': hurt_data.mean(),
            'Mean_Diff': mean_diff,
            'T_Statistic': t_stat,
            'P_Value': p_value,
            'Cohens_D': effect_size,
            'Significant': 'Yes (p<0.05)' if p_value < 0.05 else 'No (p>=0.05)',
            'Effect_Size': 'Large' if abs(effect_size) >= 0.8 else ('Medium' if abs(effect_size) >= 0.5 else 'Small')
        })

sig_df = pd.DataFrame(significance_results)
print(sig_df.to_string(index=False))

print("""
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
INTERPRETATION GUIDE:
• P-Value < 0.05: Statistically significant difference between PRF-Benefit and PRF-Hurt
• Cohen's D (Effect Size):
  - Small: |d| < 0.5  → Weak practical difference
  - Medium: 0.5 ≤ |d| < 0.8 → Moderate practical difference  
  - Large: |d| ≥ 0.8 → Strong practical difference
• Positive Mean_Diff: PRF-Benefit queries have HIGHER scores on this QPP
• Negative Mean_Diff: PRF-Benefit queries have LOWER scores on this QPP
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
""")

sig_df.to_csv('qpp_significance_tests.csv', index=False)
print("✓ Statistical test results saved: qpp_significance_tests.csv")

# ========== 8. SUMMARY ==========
print("\n" + "=" * 80)
print("ANALYSIS SUMMARY")
print("=" * 80)

best_qpp = results_df.iloc[0]
print(f"\n🏆 BEST QPP METHOD: {best_qpp['QPP']}")
print(f"   Strategy: {best_qpp['direction']}")
print(f"   F1-Score: {best_qpp['f1']:.4f}")
print(f"   Accuracy: {best_qpp['accuracy']:.4f}")
print(f"   AUC-ROC:  {best_qpp['auc_roc']:.4f}")
print(f"   Optimal Threshold: {best_qpp['threshold']:.4f}")
print(f"   Sample Size: {best_qpp['n_samples']:.0f}")

print("\n📊 PERFORMANCE ASSESSMENT:")
if best_qpp['f1'] > 0.7:
    assessment = "EXCELLENT"
    interpretation = "QPP methods can reliably predict PRF effectiveness"
    recommendation = f"{best_qpp['QPP']} is a strong baseline for your selective PRF method"
elif best_qpp['f1'] > 0.6:
    assessment = "GOOD"
    interpretation = "QPP methods have reasonable predictive power"
    recommendation = "Your selective PRF method should aim to significantly outperform this baseline"
elif best_qpp['f1'] > 0.5:
    assessment = "MODERATE"
    interpretation = "QPP methods have limited but detectable predictive ability"
    recommendation = "There is room for improvement with more sophisticated methods"
else:
    assessment = "WEAK"
    interpretation = "Simple QPP methods struggle to predict PRF effectiveness"
    recommendation = "Your selective PRF research addresses an important gap!"

print(f"   Rating: {assessment}")
print(f"   → {interpretation}")
print(f"   → {recommendation}")

print("\n💡 NEXT STEPS:")
print("   1. Compare your selective PRF method against these baselines")
print("   2. Calculate improvement: (Your_F1 - Best_Baseline_F1) / Best_Baseline_F1")
print("   3. Conduct statistical significance tests (e.g., McNemar's test)")
print("   4. Analyze query types where your method excels vs. baselines")
print("   5. Consider ensemble approaches combining multiple QPP signals")

if best_qpp['n_samples'] < 30:
    print("\n⚠️  SAMPLE SIZE WARNING:")
    print(f"   Current sample size ({best_qpp['n_samples']:.0f}) is small for robust conclusions")
    print("   Recommendations:")
    print("   - Lower threshold to include more queries (adjust BENEFIT/HURT_THRESHOLD)")
    print("   - Use cross-validation or bootstrap for stability assessment")
    print("   - Report confidence intervals in addition to point estimates")

print("\n📁 OUTPUT FILES GENERATED:")
print("   ✓ qpp_prediction_results.csv - Main performance comparison")
print("   ✓ qpp_detailed_results.csv - Comprehensive statistics") 
print("   ✓ qpp_significance_tests.csv - Statistical test results")
print("   ✓ qpp_merged_data.csv - Full dataset with all QPP scores")
print("   ✓ nqc_scores.csv, wig_scores.csv, uef_scores.csv, smv_scores.csv")

print("\n" + "=" * 80)
print("ANALYSIS COMPLETE!")
print("=" * 80)

/tmp/ipykernel_2453/694819698.py:6: DeprecationWarning: Call to deprecated function (or staticmethod) started. (use pt.java.started() instead) -- Deprecated since version 0.11.0.
  if not pt.started():


完整QPP指标计算与PRF预测分析

📊 当前阈值设置:
   PRF-Benefit: b_preference_ratio > 0.52
   PRF-Hurt:    b_preference_ratio < 0.48
   提示: 调整阈值来平衡样本量和标签质量

加载检索结果...
✓ ColBERT E2E: 310072 条结果, 43 个查询
✓ ColBERT PRF: 43000 条结果, 43 个查询
✓ 加载 preference 数据: preference.csv (43 条)
  原始列名: ['1108939', '0.913344055467992']
✓ 加载 Dense QPP: rbo_Noisyq_0.01.dl2019_queries.tsv.tsv_500 (199 条)

计算QPP指标...

1. 计算 NQC (基于E2E结果)...
   ✓ 计算完成: 43 个查询

2. 计算 WIG (E2E vs PRF差异)...
   ✓ 计算完成: 43 个查询

3. 计算 UEF (基于E2E结果)...
   ✓ 计算完成: 43 个查询

4. 计算 SMV (基于E2E结果)...
   ✓ 计算完成: 43 个查询

✓ QPP分数已保存

合并数据...
初始 preference 数据: 43 条查询
  - qid 类型: object
  - qid 样例: ['104861', '130510', '131843']

合并 NQC...
  - NQC 数据: 43 条, qid类型: object
  - NQC qid 样例: ['1037798', '104861', '1063750']
  - 合并后: 43 条, 有效 NQC: 43

合并 WIG...
  - WIG 数据: 43 条, qid类型: object
  - 合并后: 43 条, 有效 WIG: 43

合并 UEF...
  - UEF 数据: 43 条, qid类型: object
  - 合并后: 43 条, 有效 UEF: 43

合并 SMV...
  - SMV 数据: 43 条, qid类型: object
  - 合并后: 43 条, 有效 SMV: 43

合并 Dense QPP...
 